# 02 - Tabu Search


## Primera metaheuristica

Tabu Search es una metaheuristica de una sola solucion. Esto significa que el
algoritmo mantiene una solucion actual y se va moviendo desde ella hacia
soluciones vecinas.

La idea nace desde la busqueda local, pero agrega memoria. Esa memoria evita que
el algoritmo vuelva inmediatamente a decisiones recientes y quede girando sobre
la misma zona del espacio de busqueda.

Como toda metaheuristica, no garantiza encontrar el optimo global. Su objetivo
es encontrar soluciones de buena calidad usando una estrategia mas inteligente
que probar soluciones al azar o quedarse solo con mejoras inmediatas.


## Busqueda local

Para entender Tabu Search, primero hay que entender busqueda local.

Una busqueda local parte desde una solucion inicial y revisa soluciones cercanas,
llamadas vecinas. Si encuentra una vecina mejor, se mueve hacia ella. El problema
es que una busqueda local simple puede quedar atrapada en un optimo local.

Tabu Search cambia esa regla. No exige moverse siempre a una solucion mejor. En
cada iteracion elige el mejor movimiento admisible, incluso si ese movimiento
empeora temporalmente la solucion actual.

Ese detalle es importante: aceptar un empeoramiento controlado puede permitir
salir de una zona estancada y llegar despues a mejores soluciones.


## Solucion y vecindario

Una solucion es una configuracion candidata del problema.

En TSP puede ser un orden de ciudades. En mochila puede ser un vector de ceros y
unos. En scheduling puede ser un orden de tareas. La representacion cambia, pero
la logica de Tabu Search se mantiene.

El vecindario es el conjunto de soluciones que se pueden obtener aplicando un
cambio pequeno a la solucion actual.

Ese cambio se llama movimiento. En TSP puede ser invertir un segmento de una
ruta. En mochila puede ser agregar o quitar un objeto. En scheduling puede ser
intercambiar dos tareas.

La calidad del vecindario importa mucho. Si el vecindario es pobre, el algoritmo
explora poco. Si es demasiado grande, cada iteracion puede volverse costosa.


## Lista tabu

La lista tabu es la memoria de corto plazo del algoritmo.

Cuando Tabu Search hace un movimiento, guarda algun atributo de ese movimiento y
lo declara temporalmente prohibido. La idea no es prohibir soluciones para
siempre, sino evitar volver de inmediato al estado anterior.

Por ejemplo, si el algoritmo acaba de invertir cierto segmento de una ruta, puede
marcar ese movimiento como tabu durante algunas iteraciones. Asi se reduce el
riesgo de deshacer el cambio al paso siguiente.

La lista tabu no representa una memoria perfecta de todo lo visitado. Es una
memoria practica, limitada y temporal.


## Tenure

El tenure es la cantidad de iteraciones durante las cuales un movimiento queda
prohibido.

Un tenure muy corto puede permitir ciclos, porque el algoritmo olvida demasiado
rapido. Un tenure muy largo puede bloquear movimientos utiles y alejar demasiado
la busqueda de zonas prometedoras.

Por eso el tenure controla una parte importante del equilibrio entre explotacion
y exploracion. Con memoria corta se explota mas la zona actual. Con memoria mas
larga se fuerza al algoritmo a buscar caminos distintos.


## Aspiracion

El criterio de aspiracion permite romper una prohibicion tabu cuando el
movimiento produce una solucion especialmente buena.

La regla mas comun es permitir un movimiento tabu si mejora la mejor solucion
encontrada hasta el momento.

Esto evita que la memoria sea demasiado rigida. La lista tabu sirve para evitar
ciclos, pero no deberia impedir una mejora clara.


## Estructura en Python

La forma tecnica de implementar Tabu Search es separar el algoritmo del problema.
El algoritmo no necesita saber si trabaja con rutas, mochila o tareas; necesita
recibir funciones que describan como moverse y como evaluar.

En el notebook, la estructura se implementa con estas piezas:

- `initial`: solucion inicial.
- `neighborhood(solution)`: genera movimientos y soluciones vecinas.
- `objective(solution)`: mide la calidad de una solucion.
- `tenure`: duracion de la memoria tabu.
- `iterations`: numero maximo de iteraciones.
- `sense`: indica si el problema es de minimizacion o maximizacion.

Esta separacion permite estudiar Tabu Search como una metaheuristica general. El
metodo se mantiene; lo que cambia es la forma de representar soluciones y de
construir el vecindario.


In [1]:
from dataclasses import dataclass
from math import inf

@dataclass
class TabuSearchResult:
    best_solution: object
    best_value: float
    history: list
    iterations: int
    skipped_tabu: int
    accepted_worse: int


def tabu_search(
    initial,
    neighborhood,
    objective,
    *,
    tenure=10,
    iterations=100,
    sense="min",
    aspiration=True,
):
    """
    Tabu Search generico.

    neighborhood(solution) debe entregar pares (move, neighbor).
    move es el atributo que se guarda en la lista tabu.
    objective(solution) evalua la calidad de la solucion.
    """
    if sense not in {"min", "max"}:
        raise ValueError("sense debe ser 'min' o 'max'")

    sign = 1 if sense == "min" else -1

    current = initial
    current_value = objective(current)
    best_solution = current
    best_value = current_value
    best_key = sign * best_value

    tabu_until = {}
    history = [best_value]
    skipped_tabu = 0
    accepted_worse = 0
    completed_iterations = 0

    for it in range(iterations):
        candidates = []

        for move, neighbor in neighborhood(current):
            value = objective(neighbor)
            key = sign * value

            is_tabu = tabu_until.get(move, -1) > it
            improves_best = key < best_key

            if is_tabu and not (aspiration and improves_best):
                skipped_tabu += 1
                continue

            candidates.append((key, value, move, neighbor))

        if not candidates:
            break

        candidates.sort(key=lambda item: item[0])
        _, next_value, move, next_solution = candidates[0]

        if sign * next_value > sign * current_value:
            accepted_worse += 1

        current = next_solution
        current_value = next_value
        tabu_until[move] = it + tenure

        current_key = sign * current_value
        if current_key < best_key:
            best_solution = current
            best_value = current_value
            best_key = current_key

        expired = [move for move, end in tabu_until.items() if end <= it]
        for move in expired:
            del tabu_until[move]

        history.append(best_value)
        completed_iterations += 1

    return TabuSearchResult(
        best_solution=best_solution,
        best_value=best_value,
        history=history,
        iterations=completed_iterations,
        skipped_tabu=skipped_tabu,
        accepted_worse=accepted_worse,
    )

## Lectura del codigo

El codigo implementa un motor general de Tabu Search.

La clase `TabuSearchResult` ordena la salida del algoritmo. Guarda la mejor
solucion encontrada, su valor objetivo, el historial de mejora y algunas
estadisticas de la busqueda.

La funcion `tabu_search` recibe una solucion inicial, una funcion que genera
vecinos y una funcion objetivo. Con eso puede trabajar sobre distintos problemas
sin cambiar la estructura del algoritmo.

La variable `current` representa la solucion actual. La variable `best_solution`
representa la mejor solucion encontrada hasta el momento.

La estructura `tabu_until` funciona como lista tabu. Para cada movimiento guarda
hasta que iteracion ese movimiento sigue prohibido.

En cada iteracion el algoritmo revisa los vecinos de la solucion actual. Si un
movimiento es tabu, se descarta, salvo que cumpla el criterio de aspiracion y
mejore la mejor solucion global.

Luego se elige el mejor candidato admisible. Ese movimiento se acepta aunque sea
peor que la solucion actual, porque Tabu Search no es una busqueda greedy pura.
Esa aceptacion de empeoramientos controlados es una de las razones por las que
puede escapar de optimos locales.

Al final de cada iteracion se actualiza la memoria tabu, se revisa si hay un
nuevo incumbente y se guarda el historial de la mejor solucion.


## Ejemplo 1 - TSP

En TSP, una solucion es un orden de ciudades. El vecindario se construye con movimientos 2-opt: se invierte un segmento de la ruta para generar una ruta vecina.

Tabu Search guarda el movimiento realizado, por ejemplo el segmento invertido, para evitar deshacerlo inmediatamente.


In [2]:
import math

coords = [
    (0.10, 0.20), (0.25, 0.85), (0.50, 0.55),
    (0.80, 0.75), (0.90, 0.15), (0.45, 0.10),
    (0.15, 0.55), (0.65, 0.25),
]

n = len(coords)

def distance(a, b):
    ax, ay = coords[a]
    bx, by = coords[b]
    return math.hypot(ax - bx, ay - by)

def tour_length(tour):
    return sum(distance(tour[i], tour[(i + 1) % len(tour)]) for i in range(len(tour)))

def two_opt_neighborhood(tour):
    tour = tuple(tour)
    for i in range(1, n - 1):
        for j in range(i + 1, n):
            neighbor = tour[:i] + tuple(reversed(tour[i:j + 1])) + tour[j + 1:]
            move = (i, j)
            yield move, neighbor

initial = tuple(range(n))
result = tabu_search(
    initial,
    two_opt_neighborhood,
    tour_length,
    tenure=5,
    iterations=60,
    sense="min",
)

print("Ruta inicial:", initial)
print("Distancia inicial:", round(tour_length(initial), 4))
print("Mejor ruta:", result.best_solution)
print("Mejor distancia:", round(result.best_value, 4))
print("Iteraciones:", result.iterations)
print("Movimientos tabu saltados:", result.skipped_tabu)
print("Veces que acepto empeorar:", result.accepted_worse)

Ruta inicial: (0, 1, 2, 3, 4, 5, 6, 7)
Distancia inicial: 4.1554
Mejor ruta: (0, 6, 1, 2, 3, 4, 7, 5)
Mejor distancia: 2.9124
Iteraciones: 60
Movimientos tabu saltados: 230
Veces que acepto empeorar: 23


## Lectura del ejemplo TSP

El objetivo es minimizar la distancia del tour. En cada iteracion se revisan rutas vecinas y se elige la mejor admisible.

Si la mejor ruta admisible es peor que la actual, igual puede aceptarse. Esto es lo que permite salir de optimos locales.


## Ejemplo 2 - Scheduling

Scheduling es un caso donde Tabu Search suele ser util porque pequenas permutaciones del orden de tareas pueden cambiar mucho el resultado.

Aqui se usa una maquina unica con trabajos que tienen tiempo de procesamiento, fecha de entrega y peso de atraso. La solucion es un orden de trabajos y el objetivo es minimizar la tardanza ponderada total.


In [3]:
jobs = {
    "A": {"processing": 3, "due": 4,  "weight": 4},
    "B": {"processing": 5, "due": 10, "weight": 2},
    "C": {"processing": 2, "due": 6,  "weight": 6},
    "D": {"processing": 6, "due": 14, "weight": 1},
    "E": {"processing": 4, "due": 9,  "weight": 3},
    "F": {"processing": 3, "due": 12, "weight": 5},
}

def weighted_tardiness(order):
    time = 0
    total = 0
    for job in order:
        data = jobs[job]
        time += data["processing"]
        tardiness = max(0, time - data["due"])
        total += data["weight"] * tardiness
    return total

def swap_neighborhood(order):
    order = tuple(order)
    for i in range(len(order) - 1):
        for j in range(i + 1, len(order)):
            candidate = list(order)
            candidate[i], candidate[j] = candidate[j], candidate[i]
            yield (i, j), tuple(candidate)

initial = tuple(sorted(jobs, key=lambda job: jobs[job]["due"]))
result = tabu_search(
    initial,
    swap_neighborhood,
    weighted_tardiness,
    tenure=4,
    iterations=80,
    sense="min",
)

print("Orden inicial:", initial)
print("Tardanza inicial:", weighted_tardiness(initial))
print("Mejor orden:", result.best_solution)
print("Mejor tardanza ponderada:", result.best_value)
print("Veces que acepto empeorar:", result.accepted_worse)

Orden inicial: ('A', 'C', 'E', 'B', 'F', 'D')
Tardanza inicial: 42
Mejor orden: ('A', 'C', 'E', 'F', 'B', 'D')
Mejor tardanza ponderada: 23
Veces que acepto empeorar: 40


## Lectura del ejemplo Scheduling

El vecindario se construye intercambiando dos trabajos del orden actual. El movimiento tabu es el par de posiciones intercambiadas.

Este ejemplo es bueno para Tabu porque el algoritmo puede aceptar un orden temporalmente peor para escapar de una secuencia localmente buena, pero globalmente limitada.


## Parametros

Los parametros principales son el tenure, el numero de iteraciones y el
vecindario.

El tenure define cuanta memoria tiene el algoritmo. Las iteraciones definen el
presupuesto de busqueda. El vecindario define que movimientos son posibles desde
cada solucion.

En la practica, estos parametros afectan mucho el resultado. Tabu Search no es
solo "ejecutar un algoritmo"; tambien implica decidir que movimientos tienen
sentido para el problema.


## Complejidad

El costo por iteracion depende principalmente del tamano del vecindario y del
costo de evaluar cada vecino.

Si en cada iteracion se revisan muchos vecinos, el metodo puede volverse caro.
Por eso en problemas grandes a veces se evalua solo una parte del vecindario o
se usan estructuras para calcular cambios de costo mas rapido.

La complejidad total depende del numero de iteraciones multiplicado por el
trabajo necesario para evaluar los candidatos.


## Cuando se usa

Tabu Search se usa en problemas donde una busqueda local simple se queda
atrapada facilmente, pero donde es posible definir movimientos razonables entre
soluciones.

Es comun en rutas, asignacion, scheduling, seleccion de subconjuntos y otros
problemas combinatorios.

Funciona bien cuando se puede construir un buen vecindario y cuando la memoria
tabu ayuda a evitar ciclos sin bloquear demasiado la busqueda.


## Resumen

Tabu Search es una metaheuristica de una sola solucion basada en busqueda local
con memoria.

Su idea central es moverse al mejor vecino admisible, aunque no siempre mejore la
solucion actual, y usar una lista tabu para evitar volver inmediatamente sobre
movimientos recientes.

No garantiza el optimo global, pero puede escapar de optimos locales mejor que
una busqueda local greedy. Su rendimiento depende del vecindario, el tenure, el
criterio de aspiracion y el numero de iteraciones.
